# RAG Basics: Retrieval-Augmented Generation

LLMs are incredibly powerful, but they have two fundamental problems:
1. **Knowledge cutoff**: They only know what was in their training data. Ask about recent events? They hallucinate.
2. **No access to private data**: They can't see your company's documents, your research papers, or your database.

**RAG** solves both problems by giving the LLM "reading material" before it answers:

```
Without RAG:  Question → LLM → Answer (from memory, might be wrong)
With RAG:     Question → Search Documents → LLM + Context → Answer (grounded in facts)
```

### The Three Steps of RAG

1. **Retrieve**: Find the most relevant documents for the user's question
2. **Augment**: Inject those documents into the prompt as context
3. **Generate**: Let the LLM answer based on the provided context

In this notebook, we'll build a complete RAG pipeline from scratch — no fancy libraries, just Python and Ollama.

In [1]:
import ollama
import numpy as np

MODEL = 'llama3'

## 1. Our "Knowledge Base"

Let's create some private information that the LLM **absolutely cannot know** — things not in its training data. This is the key test: can RAG help the model answer correctly about data it has never seen?

We'll use a geospatial research team scenario to tie back to our curriculum theme.

In [2]:
documents = [
    "The GeoLab research team lead is Dr. Amina Malik. She specializes in flood prediction using SAR imagery and joined in 2021.",
    "Our main research lab is located in the NUST Science Building, Room 312, Islamabad.",
    "The team uses Sentinel-1 SAR data for flood mapping and Sentinel-2 optical data for land cover classification.",
    "Current project 'AquaSense' aims to deploy real-time flood early warning for Sindh province. Deadline: March 2026.",
    "The GIS server credentials: URL is gis.geolab.pk, username is 'researcher', password for guest access is 'geolab2024'.",
    "Team uses QGIS for visualization, Google Earth Engine for large-scale processing, and PyTorch for deep learning models.",
    "Monthly budget for cloud computing (GEE and AWS) is PKR 150,000. Managed by Operations Lead Hassan Raza.",
    "The team published 3 papers in 2024: two in Remote Sensing journal, one in ISPRS.",
]

print(f"Knowledge base: {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"  [{i}] {doc[:80]}...")

Knowledge base: 8 documents
  [0] The GeoLab research team lead is Dr. Amina Malik. She specializes in flood predi...
  [1] Our main research lab is located in the NUST Science Building, Room 312, Islamab...
  [2] The team uses Sentinel-1 SAR data for flood mapping and Sentinel-2 optical data ...
  [3] Current project 'AquaSense' aims to deploy real-time flood early warning for Sin...
  [4] The GIS server credentials: URL is gis.geolab.pk, username is 'researcher', pass...
  [5] Team uses QGIS for visualization, Google Earth Engine for large-scale processing...
  [6] Monthly budget for cloud computing (GEE and AWS) is PKR 150,000. Managed by Oper...
  [7] The team published 3 papers in 2024: two in Remote Sensing journal, one in ISPRS...


## 2. Creating Embeddings: Turning Text into Vectors

This is the crucial step. We need to convert text into **numbers** (vectors) so we can mathematically compare how similar two pieces of text are.

### What is an embedding?
An embedding is a list of numbers (typically 768-4096 numbers) that represents the **meaning** of a text. Texts with similar meanings will have similar vectors, even if they use completely different words.

For example:
- "dog" and "puppy" → Very similar embeddings
- "dog" and "quantum physics" → Very different embeddings

Ollama can generate embeddings using the same model we use for chat.

In [3]:
def get_embedding(text):
    """Convert text to a vector embedding using Ollama."""
    return ollama.embeddings(model=MODEL, prompt=text)['embedding']

# Create embeddings for all our documents
print("Generating embeddings for all documents...")
doc_embeddings = [get_embedding(doc) for doc in documents]

# Let's examine what an embedding looks like
print(f"\nEmbedding dimensions: {len(doc_embeddings[0])}")
print(f"First 10 values of doc[0] embedding: {doc_embeddings[0][:10]}")
print(f"\nEach document is now a vector of {len(doc_embeddings[0])} numbers!")

Generating embeddings for all documents...

Embedding dimensions: 4096
First 10 values of doc[0] embedding: [-3.1840224266052246, -4.1060590744018555, -1.3995476961135864, 1.0627021789550781, 2.671011447906494, 1.3175599575042725, -4.422698974609375, -0.6279517412185669, -3.200195789337158, 1.3602254390716553]

Each document is now a vector of 4096 numbers!


## 3. Retrieval: Finding Relevant Documents

Now comes the "search" part. When a user asks a question, we:
1. Convert their question into an embedding
2. Compare it against all document embeddings
3. Return the most similar documents

### Cosine Similarity
We use **cosine similarity** to measure how "close" two vectors are. It measures the angle between two vectors:
- **1.0** = Identical direction (very similar meaning)
- **0.0** = Perpendicular (unrelated)
- **-1.0** = Opposite direction (opposite meaning)

In [4]:
def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, top_k=2):
    """Find the top-k most relevant documents for a query."""
    query_emb = get_embedding(query)
    
    # Calculate similarity with every document
    similarities = [cosine_similarity(query_emb, doc_emb) for doc_emb in doc_embeddings]
    
    # Get indices sorted by similarity (highest first)
    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in ranked_indices:
        results.append({
            'document': documents[idx],
            'similarity': similarities[idx],
            'index': idx
        })
    return results

In [5]:
# Let's test retrieval on a few questions
test_queries = [
    "Who leads the team?",
    "What satellite data do they use?",
    "When is the project deadline?",
    "How much is the computing budget?",
]

for query in test_queries:
    results = retrieve(query, top_k=2)
    print(f"❓ Query: {query}")
    for r in results:
        print(f"   📄 [{r['similarity']:.3f}] {r['document'][:80]}...")
    print()

❓ Query: Who leads the team?
   📄 [0.496] The team uses Sentinel-1 SAR data for flood mapping and Sentinel-2 optical data ...
   📄 [0.480] Team uses QGIS for visualization, Google Earth Engine for large-scale processing...

❓ Query: What satellite data do they use?
   📄 [0.682] The team uses Sentinel-1 SAR data for flood mapping and Sentinel-2 optical data ...
   📄 [0.598] The GIS server credentials: URL is gis.geolab.pk, username is 'researcher', pass...

❓ Query: When is the project deadline?
   📄 [0.609] The GIS server credentials: URL is gis.geolab.pk, username is 'researcher', pass...
   📄 [0.607] Team uses QGIS for visualization, Google Earth Engine for large-scale processing...

❓ Query: How much is the computing budget?
   📄 [0.558] The GIS server credentials: URL is gis.geolab.pk, username is 'researcher', pass...
   📄 [0.518] Monthly budget for cloud computing (GEE and AWS) is PKR 150,000. Managed by Oper...



**Notice** how the retriever finds semantically relevant documents even when the query uses different words than the document. "Who leads the team?" matches the document about Dr. Amina Malik even though it doesn't contain the word "leads".

## 4. Augmentation & Generation: The Full Pipeline

Now we combine everything. The key is the **augmented prompt** — we inject the retrieved documents as context and instruct the model to answer ONLY based on that context.

In [6]:
def rag_query(user_question, top_k=2):
    """Complete RAG pipeline: Retrieve → Augment → Generate."""
    
    # Step 1: RETRIEVE
    retrieved = retrieve(user_question, top_k=top_k)
    context = "\n".join([r['document'] for r in retrieved])
    
    # Step 2: AUGMENT the prompt with context
    augmented_prompt = f"""Use ONLY the context below to answer the question. 
If the answer is not in the context, say "I don't have that information."

---
CONTEXT:
{context}
---

QUESTION: {user_question}

ANSWER:"""
    
    # Step 3: GENERATE
    response = ollama.generate(model=MODEL, prompt=augmented_prompt, options={'temperature': 0.2})
    
    return {
        'answer': response['response'],
        'sources': retrieved,
    }

In [7]:
# Let's try it!
result = rag_query("When is the AquaSense project deadline?")

print(f"🤖 Answer: {result['answer']}")
print(f"\n📚 Sources used:")
for s in result['sources']:
    print(f"   [{s['similarity']:.3f}] {s['document'][:80]}...")

🤖 Answer: The answer is: March 2026.

📚 Sources used:
   [0.647] Current project 'AquaSense' aims to deploy real-time flood early warning for Sin...
   [0.597] Team uses QGIS for visualization, Google Earth Engine for large-scale processing...


In [8]:
# Test with multiple questions
questions = [
    "Who is Dr. Amina Malik?",
    "What tools does the team use for visualization?",
    "Where is the lab located?",
    "What is the weather in Islamabad?"  # This is NOT in our knowledge base!
]

for q in questions:
    result = rag_query(q)
    print(f"❓ {q}")
    print(f"🤖 {result['answer'][:200]}")
    print()

❓ Who is Dr. Amina Malik?
🤖 Dr. Amina Malik is the GeoLab research team lead, specializing in flood prediction using SAR imagery and joined in 2021.

❓ What tools does the team use for visualization?
🤖 According to the context, the team uses QGIS for visualization.

❓ Where is the lab located?
🤖 The lab is located in the NUST Science Building, Room 312, Islamabad.

❓ What is the weather in Islamabad?
🤖 I don't have that information.



**That last question is the real test!** The weather in Islamabad is NOT in our knowledge base, so a good RAG system should say "I don't have that information" instead of hallucinating. This is a massive advantage over vanilla LLMs.

## 5. The Chunking Problem

Our example used short, clean documents. In the real world, you'll have long PDFs, research papers, and web pages. You can't embed an entire 50-page paper as one document — the embedding would be too "blurry" (averaging too much meaning into one vector).

The solution is **chunking**: splitting long documents into smaller, focused pieces.

### Common Chunking Strategies

| Strategy | How it Works | Best For |
| :--- | :--- | :--- |
| **Fixed-size** | Split every N characters/words | Simple, works okay |
| **Paragraph-based** | Split at paragraph boundaries | Well-structured text |
| **Sentence-based** | Split at sentence boundaries | Precise retrieval |
| **Overlap** | Fixed-size with 10-20% overlap | Prevents cutting context |

Let's implement a simple chunker with overlap:

In [9]:
def chunk_text(text, chunk_size=200, overlap=50):
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # Slide back by overlap amount
    return chunks

# Example with a longer text
long_text = """Remote sensing is the acquisition of information about an object or phenomenon 
without making physical contact with it. In the context of Earth observation, this primarily 
involves satellites collecting data about Earth's surface. There are two main types: passive 
sensors that detect natural radiation (like optical cameras), and active sensors that emit 
their own signal and measure the reflection (like SAR radar). The spatial resolution of a 
sensor determines the smallest object it can detect. Sentinel-2 has 10m resolution for 
visible bands, while commercial satellites like Maxar's WorldView can achieve 30cm resolution."""

chunks = chunk_text(long_text, chunk_size=30, overlap=5)
print(f"Original: {len(long_text.split())} words")
print(f"Chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: ({len(chunk.split())} words) {chunk[:80]}...")

Original: 92 words
Chunks: 4

  Chunk 0: (30 words) Remote sensing is the acquisition of information about an object or phenomenon w...
  Chunk 1: (30 words) primarily involves satellites collecting data about Earth's surface. There are t...
  Chunk 2: (30 words) that emit their own signal and measure the reflection (like SAR radar). The spat...
  Chunk 3: (17 words) Sentinel-2 has 10m resolution for visible bands, while commercial satellites lik...


## 6. What's Next: From Mini-RAG to Production

What we built is a functional RAG system, but a production system would add several layers:

### Vector Databases
We stored embeddings in a Python list — fine for 8 documents, but terrible for 10,000. Real systems use specialized databases:
- **ChromaDB**: Easy to use, great for prototyping, runs locally
- **FAISS** (Facebook): Blazing fast similarity search, used by many tech companies
- **Pinecone**: Cloud-hosted, fully managed, scales automatically
- **Weaviate**: Open-source, supports hybrid search (vector + keyword)

### Advanced RAG Techniques
- **Hybrid Search**: Combine vector similarity with traditional keyword search (BM25)
- **Re-ranking**: Use a second model to re-rank retrieved documents for relevance
- **Query Expansion**: Use the LLM to rephrase the query for better retrieval
- **Multi-hop RAG**: Retrieve → Answer partially → Retrieve more → Answer fully

### Evaluation
How do you know your RAG system is working well?
- **Retrieval accuracy**: Are you finding the right documents? (Precision/Recall)
- **Answer faithfulness**: Is the answer actually supported by the retrieved context?
- **Answer relevance**: Does the answer actually address the question?

## Summary

You built a complete RAG pipeline from scratch! Let's recap:

```
1. EMBED:     Documents → Vectors (numbers representing meaning)
2. RETRIEVE:  Query → Vector → Cosine Similarity → Top-K Documents
3. AUGMENT:   Prompt = Instructions + Retrieved Context + User Question
4. GENERATE:  LLM reads the augmented prompt → Grounded answer
```

RAG is arguably the most important technique in applied LLM engineering today. It's how enterprises make LLMs useful for their specific data without the cost and complexity of fine-tuning.

## Challenge! 🚀

1. Add 5 more documents to the knowledge base about a topic you care about and test the retrieval.
2. Try changing `top_k` from 1 to 3 — does using more context improve the answers?
3. **Advanced**: Load a real text file (like a README or a paper) and use the chunker to create a knowledge base from it.

---

**Congratulations!** 🎉 You've completed the core Module 5 notebooks. Next, check out the specialized notebooks on [Fine-tuning](./fine_tuning_lora.ipynb) and [Building a Chatbot](./chatbot_with_memory.ipynb)!